# Accuracy and the confusion matrix

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
warnings.simplefilter("ignore")

## Heart disease: two models

In [ ]:
df = pd.read_csv("data/heart.csv")
print(df.shape, df["target"].value_counts().to_dict())
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, :-1], df.iloc[:, -1], test_size=0.2, random_state=2)
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
dt = DecisionTreeClassifier(random_state=1).fit(X_train, y_train)
for name, m in [("logistic regression", lr), ("decision tree", dt)]:
    y_pred = m.predict(X_test)
    print(name, "accuracy", round(accuracy_score(y_test, y_pred), 4))
    print(pd.DataFrame(confusion_matrix(y_test, y_pred), index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"]))

## Accuracy from the confusion matrix: (TP + TN) / total

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, lr.predict(X_test)).ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}  accuracy = {(tp + tn) / (tp + tn + fp + fn):.4f}")

## More than two classes

In [ ]:
for loader in (load_iris, load_digits):
    X, y = loader(return_X_y=True)
    a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=1)
    p = LogisticRegression(max_iter=10000).fit(a, c).predict(b)
    print(loader.__name__, "accuracy", round(accuracy_score(d, p), 4))
    print(confusion_matrix(d, p))

## Accuracy can mislead on imbalanced data

In [ ]:
y_true = np.r_[np.ones(10), np.zeros(99_990)]     # 10 positives in 100,000
y_pred = np.zeros(100_000)                          # a 'model' that always says 0
print("accuracy", accuracy_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))